In [2]:

import numpy as np
import pandas as pd
import requests
from lib.odds import Odds

%load_ext magic_duckdb

In [53]:
df = Odds().get_sports()

In [2]:
df = Odds().get_nba_champ_odds()

Number of events: 1

In [58]:
display(df.query("group == 'Basketball'"))

,key,group,title,description,active,has_outrights
5,basketball_euroleague,Basketball,Basketball Euroleague,Basketball Euroleague,True,False
6,basketball_nba,Basketball,NBA,US Basketball,True,False
7,basketball_nba_championship_winner,Basketball,NBA Championship Winner,Championship Winner 2024/2025,True,True
8,basketball_ncaab,Basketball,NCAAB,US College Basketball,True,False
9,basketball_ncaab_championship_winner,Basketball,NCAAB Championship Winner,US College Basketball Championship Winner,True,True


In [7]:
Odds.append_to_table(df, 'nba_champ_odds')

AttributeError: type object 'Odds' has no attribute 'append_to_table'

In [30]:
from datetime import datetime, timedelta

# Get current ISO time
current_iso_time = datetime.now().replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")

print("Current ISO Time: ", current_iso_time)

# Calculate 20 hours from now
future_time = datetime.now() + timedelta(hours=20)
future_iso_time = future_time.replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")

print("20 Hours From Now (ISO Time): ", future_iso_time)


Current ISO Time:  2024-11-11T13:49:33Z
20 Hours From Now (ISO Time):  2024-11-12T09:49:33Z


In [31]:


sport = 'basketball_nba'
odds_response = requests.get(
    f'https://api.the-odds-api.com/v4/sports/{sport}/odds',
    params={
        'api_key': '2b1fbcaa5cf9bbfca66c986129524664',
        'regions': 'us',
        'markets': 'h2h',
        'oddsFormat': 'american',
        'dateFormat': 'iso',
        'bookmakers': ['betmgm', 'fanduel', 'draftkings'],
        'commenceTimeFrom': current_iso_time,
        'commenceTimeTo': future_iso_time
    }
)

if odds_response.status_code != 200:
    print(f'Failed to get odds: status_code {odds_response.status_code}, response body {odds_response.text}')

else:
    odds_json = odds_response.json()
    print('Number of events:', len(odds_json))
    print(odds_json)

    # Check the usage quota
    print('Remaining requests', odds_response.headers['x-requests-remaining'])
    print('Used requests', odds_response.headers['x-requests-used'])

Number of events: 5
[{'id': '4c65fb969ac80375654b3bf008db7e8a', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2024-11-12T01:10:00Z', 'home_team': 'New Orleans Pelicans', 'away_team': 'Brooklyn Nets', 'bookmakers': [{'key': 'betmgm', 'title': 'BetMGM', 'last_update': '2024-11-11T20:49:15Z', 'markets': [{'key': 'h2h', 'last_update': '2024-11-11T20:49:15Z', 'outcomes': [{'name': 'Brooklyn Nets', 'price': -120}, {'name': 'New Orleans Pelicans', 'price': 100}]}]}]}, {'id': 'f6332b5b82fa9a4e4b1e2be7091171cc', 'sport_key': 'basketball_nba', 'sport_title': 'NBA', 'commence_time': '2024-11-12T01:10:00Z', 'home_team': 'Chicago Bulls', 'away_team': 'Cleveland Cavaliers', 'bookmakers': [{'key': 'betmgm', 'title': 'BetMGM', 'last_update': '2024-11-11T20:49:15Z', 'markets': [{'key': 'h2h', 'last_update': '2024-11-11T20:49:15Z', 'outcomes': [{'name': 'Chicago Bulls', 'price': 280}, {'name': 'Cleveland Cavaliers', 'price': -350}]}]}]}, {'id': 'ce9117fa84821ae069a20238f90206a3'

In [32]:
odds_meta = [
    ['id'],
    # ['has_outrights'],
    ['sport_key'],
    ['sport_title'],
    ['commence_time'],
    ['home_team'],
    ['away_team'],
    ['bookmakers', 'key'],
    ['bookmakers', 'title'],
    ['bookmakers', 'last_update'],
    ['bookmakers', 'markets', 'key'],
    ['bookmakers', 'markets', 'last_update'],
]

df = pd.json_normalize(odds_json, record_path=['bookmakers', 'markets', 'outcomes'], 
            meta=odds_meta, sep='_'
            )

In [33]:
display(df)

,name,price,id,sport_key,sport_title,commence_time,home_team,away_team,bookmakers_key,bookmakers_title,bookmakers_last_update,bookmakers_markets_key,bookmakers_markets_last_update
0,Brooklyn Nets,-120,4c65fb969ac80375654b3bf008db7e8a,basketball_nba,NBA,2024-11-12T01:10:00Z,New Orleans Pelicans,Brooklyn Nets,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
1,New Orleans Pelicans,100,4c65fb969ac80375654b3bf008db7e8a,basketball_nba,NBA,2024-11-12T01:10:00Z,New Orleans Pelicans,Brooklyn Nets,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
2,Chicago Bulls,280,f6332b5b82fa9a4e4b1e2be7091171cc,basketball_nba,NBA,2024-11-12T01:10:00Z,Chicago Bulls,Cleveland Cavaliers,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
3,Cleveland Cavaliers,-350,f6332b5b82fa9a4e4b1e2be7091171cc,basketball_nba,NBA,2024-11-12T01:10:00Z,Chicago Bulls,Cleveland Cavaliers,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
4,Houston Rockets,-800,ce9117fa84821ae069a20238f90206a3,basketball_nba,NBA,2024-11-12T01:10:00Z,Houston Rockets,Washington Wizards,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
5,Washington Wizards,550,ce9117fa84821ae069a20238f90206a3,basketball_nba,NBA,2024-11-12T01:10:00Z,Houston Rockets,Washington Wizards,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
6,Los Angeles Clippers,220,aac06078dcea994124e8ba33553dab22,basketball_nba,NBA,2024-11-12T01:10:00Z,Oklahoma City Thunder,Los Angeles Clippers,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
7,Oklahoma City Thunder,-275,aac06078dcea994124e8ba33553dab22,basketball_nba,NBA,2024-11-12T01:10:00Z,Oklahoma City Thunder,Los Angeles Clippers,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
8,Sacramento Kings,-115,a03bb6d54e8c501291594c78c582efe3,basketball_nba,NBA,2024-11-12T01:10:00Z,San Antonio Spurs,Sacramento Kings,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
9,San Antonio Spurs,-105,a03bb6d54e8c501291594c78c582efe3,basketball_nba,NBA,2024-11-12T01:10:00Z,San Antonio Spurs,Sacramento Kings,betmgm,BetMGM,2024-11-11T20:49:15Z,h2h,2024-11-11T20:49:15Z
